In [0]:
%pip install geopandas shapely

In [0]:
%restart_python

In [0]:
import json
import geopandas as gpd
import pandas as pd
from shapely.geometry import box
from datetime import datetime

BRONZE_FIRE_PATH = (
    "abfss://wildfire-data@stscewildfirevr01.dfs.core.windows.net/"
    "bronze/fire_history"
)

SILVER_FIRE_PATH = (
    "abfss://wildfire-data@stscewildfirevr01.dfs.core.windows.net/"
    "silver/fire_history"
)

MIN_LONGITUDE = -118.95
MIN_LATITUDE = 33.95
MAX_LONGITUDE = -118.45
MAX_LATITUDE = 34.25

CURRENT_YEAR = datetime.now().year
PROJECTED_CRS = "EPSG:3310"

print("Current year:", CURRENT_YEAR)

In [0]:
folders = dbutils.fs.ls(BRONZE_FIRE_PATH)

load_folders = sorted(
    [x.path for x in folders if "load_timestamp=" in x.path]
)

latest_fire_path = load_folders[-1]

print("Latest Bronze fire path:")
print(latest_fire_path)

In [0]:
raw_fire_df = spark.read.format("delta").load(latest_fire_path)

raw_json = raw_fire_df.select("raw_json").first()["raw_json"]
fire_json = json.loads(raw_json)

print("Raw fire features:", len(fire_json["features"]))

In [0]:
fires = gpd.GeoDataFrame.from_features(
    fire_json["features"],
    crs="EPSG:4326"
)

print("Rows:", len(fires))
print("Columns:", fires.columns.tolist())

In [0]:
fires["YEAR_"] = pd.to_numeric(
    fires["YEAR_"],
    errors="coerce"
)

fires["GIS_ACRES"] = pd.to_numeric(
    fires["GIS_ACRES"],
    errors="coerce"
)

fires = fires[
    fires["YEAR_"].between(
        2000,
        CURRENT_YEAR,
        inclusive="both"
    )
].copy()

fires["geometry"] = fires.geometry.make_valid()

fires = fires[
    fires.geometry.notna() &
    ~fires.geometry.is_empty
].copy()

print("Rows after year + geometry cleaning:", len(fires))

In [0]:
study_boundary = gpd.GeoDataFrame(
    {"study_area": ["Western Los Angeles County"]},
    geometry=[
        box(
            MIN_LONGITUDE,
            MIN_LATITUDE,
            MAX_LONGITUDE,
            MAX_LATITUDE
        )
    ],
    crs="EPSG:4326"
)

fires = gpd.clip(
    fires,
    study_boundary
)

fires = fires[
    fires.geometry.notna() &
    ~fires.geometry.is_empty
].copy()

print("Rows after Western LA clip:", len(fires))

In [0]:
fires_projected = fires.to_crs(PROJECTED_CRS)

fires["study_area_acres"] = (
    fires_projected.geometry.area /
    4046.8564224
)

fires["fire_id"] = fires["OBJECTID"].astype(str)

print("Historical fire rows:", len(fires))
print("Unique fire IDs:", fires["fire_id"].nunique())
print(
    "Fire year range:",
    int(fires["YEAR_"].min()),
    "to",
    int(fires["YEAR_"].max())
)
print("Invalid geometries:", (~fires.geometry.is_valid).sum())
print("Missing geometry:", fires.geometry.isna().sum())
print(
    "Negative study-area acres:",
    (fires["study_area_acres"] < 0).sum()
)

In [0]:
silver_fires = fires[
    [
        "fire_id",
        "OBJECTID",
        "YEAR_",
        "FIRE_NAME",
        "ALARM_DATE",
        "CONT_DATE",
        "CAUSE",
        "GIS_ACRES",
        "AGENCY",
        "DECADES",
        "study_area_acres",
        "geometry"
    ]
].copy()

silver_fires = silver_fires.rename(
    columns={
        "OBJECTID": "object_id",
        "YEAR_": "fire_year",
        "FIRE_NAME": "fire_name",
        "ALARM_DATE": "alarm_date",
        "CONT_DATE": "containment_date",
        "CAUSE": "cause",
        "GIS_ACRES": "source_gis_acres",
        "AGENCY": "agency",
        "DECADES": "decade"
    }
)

silver_fires["fire_year"] = silver_fires["fire_year"].astype(int)

silver_fires["geometry_wkt"] = silver_fires.geometry.apply(
    lambda geom: geom.wkt if geom is not None else None
)

silver_fires = silver_fires.drop(columns=["geometry"])

print("Silver fire rows:", len(silver_fires))
print(silver_fires.head())

In [0]:
silver_fire_spark = spark.createDataFrame(silver_fires)

silver_fire_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .save(SILVER_FIRE_PATH)

print("Silver fire history written successfully")
print("Rows written:", silver_fire_spark.count())
print("Path:", SILVER_FIRE_PATH)

In [0]:
check_fires = spark.read.format("delta").load(SILVER_FIRE_PATH)

print("Total rows:", check_fires.count())

print(
    "Unique fire IDs:",
    check_fires.select("fire_id").distinct().count()
)

year_stats = check_fires.agg(
    {"fire_year": "min"}
).first()[0], check_fires.agg(
    {"fire_year": "max"}
).first()[0]

print("Fire year range:", year_stats[0], "to", year_stats[1])